In [ ]:
import os
import sys

from maxtext.utils.globals import MAXTEXT_REPO_ROOT

# Add the project root to the system path if it's not already there
if MAXTEXT_REPO_ROOT not in sys.path:
  sys.path.insert(0, MAXTEXT_REPO_ROOT)
  print(f"Added '{MAXTEXT_REPO_ROOT}' to sys.path")

In [ ]:
import maxtext as mt
from maxtext.configs import pyconfig
import numpy as np
from maxtext.input_pipeline import input_pipeline_utils
import os
from maxtext.common import common_types
import jax
from maxtext.inference import inference_utils
from maxtext.utils import max_logging
from maxtext.utils import maxtext_utils

In [ ]:
from maxtext.utils.globals import MAXTEXT_PKG_DIR

config = pyconfig.initialize(
    [os.path.join(MAXTEXT_PKG_DIR, "decode.py"), os.path.join(MAXTEXT_PKG_DIR, "configs", "base.yml")],
    per_device_batch_size=1.0,
    run_name="test",
    enable_checkpointing=False,
    base_num_decoder_layers=2,
    max_target_length=4,
    base_emb_dim=256,
    base_num_query_heads=2,
    base_num_kv_heads=2,
    max_prefill_predict_length=4,
    # tokenizer_path="assets/tokenizers/llama3.1-tokenizer/",
    # model_name="llama3.1-7b",
)

model = mt.from_config(config)
mesh = model.mesh
init_rng = jax.random.PRNGKey(config.init_weights_seed)
state, _ = maxtext_utils.setup_decode_state(model, config, init_rng, mesh, None)

In [ ]:
from maxtext.utils.globals import MAXTEXT_ASSETS_ROOT

source_tokenizer = _input_pipeline_utils.get_tokenizer(
    os.path.join(MAXTEXT_ASSETS_ROOT, "tokenizers", "tokenizer_llama3.tiktoken"),
    "tiktoken",
    add_bos=True,
    add_eos=False,
)


# TODO: @mazumdera: any way to geto segment and position ids like HF tokenizer gives us?
input_ids = source_tokenizer.encode(config.prompt)  # .numpy()
ids = np.asarray(input_ids, dtype=np.int32)
s = (config.global_batch_size_to_train_on, config.max_target_length)
decoder_segment_ids = np.zeros(s) + common_types.DECODING_ACTIVE_SEQUENCE_INDICATOR
decoder_positions = np.stack(
    [np.arange(config.max_target_length, dtype=np.int32) for _ in range(config.global_batch_size_to_train_on)]
)

# TODO: @mazumdera: simplify this config.global_batch_size_to_train_on=1
ids = np.stack([ids for _ in range(config.global_batch_size_to_train_on)])
max_logging.log(
    f"input_ids={input_ids}, ids={ids}, decoder_segment_ids = {decoder_segment_ids}, decoder_positions= {decoder_positions}"
)

In [ ]:
import jax

!export TPU_LIBRARY_PATH=/home/mazumdera/.local/lib/python3.10/site-packages/libtpu/libtpu.so

jax.devices()

In [ ]:
!ls /home/mazumdera/.local/lib/python3.10/site-packages/libtpu/libtpu.so

In [ ]:
import jax.experimental.multihost_utils

full_train_logits = model.apply(
    state.params,
    ids,
    decoder_positions,
    decoder_segment_ids,
    enable_dropout=False,
    rngs={"aqt": init_rng},
)
full_train_logits = jax.experimental.multihost_utils.process_allgather(full_train_logits)
max_logging.log(f"{full_train_logits[0, 0, :]=}")

In [ ]:
selected_logits = jax.lax.dynamic_slice(
    full_train_logits, (0, 0, full_train_logits.shape[2] - 1, 0), (1, 1, 1, full_train_logits.shape[3])
)

In [ ]:
init_rng, new_rng = jax.random.split(init_rng)
first_generated_token = inference_utils.sampling(
    selected_logits,
    new_rng,
    config.decode_sampling_strategy,
    topk=config.decode_sampling_top_k,
    nucleus_topp=config.decode_sampling_nucleus_p,
    temperature=config.decode_sampling_temperature,
)

In [ ]:
first_generated_token.item()

In [ ]:
source_tokenizer.decode([first_generated_token.item()])